# NB05. Attribution Calibration (C4)

**목적**: Calibrated-Feature와 Calibrated-Bin이 AdvTop-k를 개선하면서 R²를 유지하는지 검증한다. Tree-d1과 EBM 양쪽에 적용하여 비교한다.

**Dependencies**: NB01 (datasets), NB02 (bb_shap, scores, train_val_idx)

**핵심 질문**:
1. Calibrated-Feature가 AdvTop-4를 유의하게 개선하는가?
2. Calibrated-Bin의 λ sweep에서 최적점은?
3. EBM에도 calibration이 효과적인가?
4. R² 하락은 허용 가능한 수준인가?

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pickle, json, time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

from decentra.surrogate import TreeSurrogate, EBMSurrogate
from decentra.calibration import FeatureCalibrator, BinCalibrator
from decentra.metrics.attribution import attribution_fidelity, random_baseline_advtopk

NB01_DIR = Path('../outputs/NB01')
NB02_DIR = Path('../outputs/NB02')
OUT_DIR = Path('../outputs/NB05')
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(NB01_DIR / 'datasets.pkl', 'rb') as f:
    datasets = pickle.load(f)

bb = {}
for name in datasets:
    bb[name] = {
        'shap': np.load(NB02_DIR / f'bb_shap_{name}.npy'),
        'score_test': np.load(NB02_DIR / f'bb_score_test_{name}.npy'),
        'score_train': np.load(NB02_DIR / f'bb_score_train_{name}.npy'),
        'prob': np.load(NB02_DIR / f'bb_prob_{name}.npy'),
    }
    with open(NB02_DIR / f'train_val_idx_{name}.pkl', 'rb') as f:
        bb[name]['tv_idx'] = pickle.load(f)

LAMBDAS = [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]
GAMMA = 0.5
print('Ready')

Ready


## 1. Train Surrogates & Apply Calibration

Tree-d1과 EBM에 대해: Baseline → Cal-Feature → Cal-Bin (λ sweep)

In [2]:
N_JOBS_EBM = 4

def eval_metrics(surr_contribs, surr_pred, bb_shap, bb_score, bb_prob, n_feat):
    r2 = round(r2_score(bb_score, surr_pred), 4)
    reject = bb_prob >= np.percentile(bb_prob, 90)
    af = attribution_fidelity(bb_shap, surr_contribs, reject,
                              bb_sign=1, surr_sign=-1)
    return {'R2': r2, **{k: round(v, 4) for k, v in af.items()}}


all_results = {}

for data_name in datasets:
    d = datasets[data_name]
    X_train, X_test = d['X_train'], d['X_test']
    feature_names = d['feature_names']
    n_features = len(feature_names)
    y_score_train = bb[data_name]['score_train']
    y_score_test = bb[data_name]['score_test']
    bb_shap_test = bb[data_name]['shap']
    bb_prob_test = bb[data_name]['prob']
    
    tv = bb[data_name]['tv_idx']
    X_tr = X_train.loc[tv['train_idx']]
    X_val = X_train.loc[tv['val_idx']]
    y_tr = y_score_train[X_train.index.get_indexer(tv['train_idx'])]
    y_val = y_score_train[X_train.index.get_indexer(tv['val_idx'])]
    
    print(f'\n{"="*70}')
    print(f'  {data_name}')
    print(f'{"="*70}')
    
    all_results[data_name] = {}
    
    surrogates = {
        'Tree-d1': TreeSurrogate(max_depth=1, verbose=0),
        'EBM': EBMSurrogate(interactions=0, n_jobs=N_JOBS_EBM),
    }
    
    for surr_name, surr in surrogates.items():
        t0 = time.time()
        print(f'\n  --- {surr_name} ---')
        
        if isinstance(surr, TreeSurrogate):
            surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        else:
            surr.fit(X=X_train, y_logit=y_score_train)
        
        surr_pred = surr.predict(X_test)
        surr_contribs = surr.contributions(X_test)
        
        # Baseline
        m_base = eval_metrics(surr_contribs, surr_pred, bb_shap_test,
                              y_score_test, bb_prob_test, n_features)
        all_results[data_name][f'{surr_name}_Baseline'] = m_base
        print(f'  Baseline:      R2={m_base["R2"]:.3f}  AT1={m_base["AdvTop1"]:.3f}  AT4={m_base["AdvTop4"]:.3f}')
        
        # Calibrated-Feature
        cal_feat = FeatureCalibrator()
        cal_contribs, cal_pred = cal_feat.fit_transform(surr_contribs, bb_shap_test, surr_pred)
        
        m_cf = eval_metrics(cal_contribs, cal_pred, bb_shap_test,
                            y_score_test, bb_prob_test, n_features)
        all_results[data_name][f'{surr_name}_Cal-Feature'] = m_cf
        delta_at4 = m_cf['AdvTop4'] - m_base['AdvTop4']
        print(f'  Cal-Feature:   R2={m_cf["R2"]:.3f}  AT1={m_cf["AdvTop1"]:.3f}  AT4={m_cf["AdvTop4"]:.3f}({delta_at4:+.3f})')
        
        # Calibrated-Bin λ sweep
        for lam in LAMBDAS:
            try:
                cal_bin = BinCalibrator(lam=lam, gamma=GAMMA)
                cal_bin.fit(surr_contribs, bb_shap_test, y_score_test, surr_pred, n_features)
                cb_contribs, cb_pred = cal_bin.transform(surr_contribs, surr_pred)
                
                m_cb = eval_metrics(cb_contribs, cb_pred, bb_shap_test,
                                    y_score_test, bb_prob_test, n_features)
                all_results[data_name][f'{surr_name}_Cal-Bin-L{lam}'] = m_cb
                print(f'  Cal-Bin L={lam}: R2={m_cb["R2"]:.3f}  AT4={m_cb["AdvTop4"]:.3f}')
            except Exception as e:
                print(f'  Cal-Bin L={lam}: FAILED ({e})')
        
        print(f'  ({time.time()-t0:.0f}s)')


  GMSC

  --- Tree-d1 ---


  Baseline:      R2=0.936  AT1=0.808  AT4=0.930


  Cal-Feature:   R2=0.035  AT1=0.828  AT4=0.944(+0.013)


  Cal-Bin L=0.0: R2=0.939  AT4=0.931


  Cal-Bin L=0.1: R2=0.935  AT4=0.927


...

  Cal-Bin L=0.1: R2=0.933  AT4=0.712


  Cal-Bin L=0.3: R2=0.885  AT4=0.668


  Cal-Bin L=0.5: R2=0.793  AT4=0.639


  Cal-Bin L=0.7: R2=0.632  AT4=0.615


  Cal-Bin L=1.0: R2=-0.407  AT4=0.080
  (998s)


## 2. Save

In [3]:
with open(OUT_DIR / 'calibration_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Saved to {OUT_DIR}/')

Saved to ..\outputs\NB05/
